# US import/tariff exposure (optional observed trade data)

Only report effective collected duties when imports and duties have the same underlying classification, partner set and time basis.


In [1]:
from pathlib import Path
import sys
for loc in [Path.cwd(),Path.cwd()/'notebooks',Path.cwd()/'research_expansion'/'notebooks']:
    if (loc/'common.py').exists():
        sys.path.insert(0,str(loc.resolve()));break
from common import root, output


In [2]:
import pandas as pd,numpy as np
path=root()/'data'/'processed'/'tariff_exposure.csv'
if not path.exists():
    output('tariff_exposure',{'model':'Tariff exposure descriptive audit','status':'skipped','reason':'No verified observed USITC aligned trade/duty dataset at data/processed/tariff_exposure.csv'})
else:
    t=pd.read_csv(path,dtype={'country':'string','industry_code':'string'})
    required={'month','country','industry_code','import_value_usd','duties_collected_usd','source_url','classification_method'}
    if not required.issubset(t.columns):
        raise ValueError('Tariff exposure missing fields: '+str(sorted(required-set(t.columns))))
    t['month']=pd.to_datetime(t.month,errors='coerce')
    for col in ['import_value_usd','duties_collected_usd']:
        t[col]=pd.to_numeric(t[col],errors='coerce')
    t=t.dropna(subset=['month','country','industry_code','import_value_usd','duties_collected_usd','source_url'])
    t=t[(t['import_value_usd']>0)&(t['duties_collected_usd']>=0)]
    if t.empty:
        output('tariff_exposure',{'status':'skipped','reason':'No validated nonnegative duties and positive comparable import values'})
    else:
        t['period']=np.select([t.month.between('2018-07-01','2019-12-31'),t.month.between('2025-04-01','2026-01-31'),t.month.between('2026-03-01','2026-06-30'),t.month.between('2026-08-01','2026-10-31')],['2018_19','2025','2026_sec122','2026_later'],default='other')
        s=t.groupby(['country','industry_code','period'],dropna=False).agg(import_usd=('import_value_usd','sum'),duties_usd=('duties_collected_usd','sum'),n_rows=('month','count')).reset_index()
        s['effective_duty_pct']=100*s.duties_usd/s.import_usd
        output('tariff_exposure',{'model':'observed customs duties / observed imports','status':'descriptive','n_rows':len(t),
            'summary':s.to_dict('records'),'limitations':['Observed effective duties are NOT statutory tariff rates','Tariff periods changed in February and July 2026','Not an estimate of causal effect on Indian or German production','Monthly rows with effective policy changes inside a month must be separately marked']})
        display(s)


Saved C:\Users\mehak\OneDrive\ドキュメント\ChatGPT\Industry Res\reports\research\models\tariff_exposure.json
